# 단계 4 — 미다스(`.mgt`) 파서 도구 만들기

**`structural-mcp` 누적 빌드업의 단계 ④**: 단계 ③에서 만든 강도 검토 도구에 이어, 외부 구조해석 소프트웨어가 만들어낸 모델 파일을 직접 읽어들이는 도구를 추가합니다.

## 본 노트북의 위치
이 노트북은 강의노트 7주차 본문 §2.7 단계 ③ 라인 1484에 명시된 미다스 파서 시그니처를 그대로 구현하는 단계입니다. 단계 1 노트북에서 만든 FastMCP 인스턴스, 단계 2 노트북에서 만든 KDS 요약 리소스, 단계 3 노트북에서 만든 휨 검토 도구와 전단 검토 도구를 모두 가지고 시작합니다. 본 단계의 산출물은 도구 세 개와 리소스 한 개를 갖춘 통합 `structural_mcp.py` 파일이며, 다음 단계 5에서 그대로 가져다 쓸 수 있도록 마지막 셀에서 다시 저장됩니다.

## 학습 목표
이번 단계에서는 **한국에서 가장 널리 쓰이는 구조해석 소프트웨어 중 하나인 미다스 시빌(Midas Civil)·미다스 젠(Midas Gen)이 사용하는 모델 파일**을 클로드가 직접 읽을 수 있는 형태의 도구로 노출합니다. 정적 데이터를 보여주는 리소스와 동적인 액션을 수행하는 도구의 차이를 실제 코드로 구분하여 익히고, 강의노트에 명시된 시그니처를 충실히 구현합니다. 또한 강의노트가 거듭 강조하는 절대경로 사용 원칙과 한국어 도메인 용어 보존 원칙을 코드로 직접 적용해 봅니다.

## 선행 학습 사항
단계 1부터 단계 3까지 누적된 `structural_mcp.py` 파일이 손에 있어야 합니다. 파이썬 3.11 이상과 모델 컨텍스트 프로토콜 SDK, 그리고 데이터 검증을 위한 파이댄틱이 설치되어 있어야 합니다. 검증용 미다스 모델 파일이 따로 없어도 본 노트북 §4에서 자동으로 만들어주므로 걱정하지 않아도 됩니다.

## 본 단계 이후의 흐름
본 노트북을 마치고 나면 단계 5에서 도메인 워크플로 자체를 캡슐화한 프롬프트를 추가하고, 단계 6에서 클로드 코드 명령행 인터페이스에 등록하여 자연어로 직접 사용해 보는 흐름으로 이어집니다. 즉 본 단계는 6단계 누적 빌드업의 정중앙에 해당하며, 이후 단계는 본 노트북의 산출물 위에 차곡차곡 쌓아 올리는 구조입니다.


## §1. 미다스 파일 파서를 왜 도구로 노출하는가

구조해석 결과를 대형언어모델이 직접 읽고 검토하려면 두 가지 설계 선택지가 있습니다. 정적인 데이터로 노출할 것인가, 아니면 동적인 액션으로 노출할 것인가는 모델 컨텍스트 프로토콜 설계의 가장 중요한 갈림길입니다.

리소스(Resource)로 노출하면 고정된 식별자(URI)에 정적인 데이터를 매달아 두는 형태가 됩니다. 그러나 이 방식은 파일 경로마다 식별자를 미리 등록해야 하므로 사용자가 임의의 파일 경로를 처리할 수 없다는 한계가 있습니다.

도구(Tool)로 노출하면 인자로 파일 경로를 받아 동적으로 파싱할 수 있습니다. 사용자가 자신의 컴퓨터에 있는 어떤 미다스 모델 파일이든 분석할 수 있게 되므로, 본 강의에서는 이 방식을 채택합니다.

> [!finding] 강의노트가 강조하는 핵심 원칙
> "리소스는 데이터를 노출하고, 도구는 액션을 수행한다"는 원칙을 따라야 합니다. 파일 시스템에서 *경로를 입력으로 받아 파싱하는 동작*은 **상태를 읽어들이는 액션**이므로 도구로 노출하는 것이 자연스럽습니다. 만약 모델 식별자가 고정된 백엔드라면, 즉 데이터베이스에 저장된 모델이라면, 형판형 리소스로 노출할 수도 있습니다 (강의노트 §2.7 머메이드 다이어그램 1499행 참조).

> [!tip] 도메인 관점에서 본 의미
> 미다스 시빌과 미다스 젠은 한국 구조설계 사무소에서 가장 보편적으로 쓰이는 해석 소프트웨어입니다. 모델 파일은 그 모델을 **텍스트 명령 형식**으로 직렬화한 결과입니다. 즉 클로드가 이 파일을 읽을 수 있다는 것은 **실무자가 작성한 모델 자체를 그대로 대형언어모델에게 전달**할 수 있다는 의미가 됩니다.


## §2. 환경 준비 — 이전 단계 산출물 다시 불러오기

단계 3까지 만든 도구와 리소스를 그대로 재사용하기 위해 같은 모듈을 (또는 동등한 코드를) 다시 정의합니다. 본 노트북은 누적 빌드업 방식을 따르므로, 단계 4의 마지막 셀에서 통합본을 다시 저장하여 다음 단계에서 그대로 가져다 쓸 수 있게 합니다.

In [ ]:
# Week_07.md §2.7 — 누적 빌드업: Stage 3까지 만든 structural_mcp.py를 그대로 import.
# (실습 환경에서는 직접 같은 디렉토리에서 재정의해도 동작)
import json
import math
import os
from pathlib import Path

from pydantic import Field
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("StructuralMCP", log_level="ERROR")

# ── (Stage 3 재현) Tools — 휨/전단 강도 검토 ───────────────────────
@mcp.tool(
    name="check_flexural_strength",
    description="RC 보의 휨 강도를 KDS 41 17 00 기준으로 검토 (SI: mm·MPa·kN·m).",
)
def check_flexural_strength(
    b: float = Field(description="보 폭 (mm)"),
    d: float = Field(description="유효 깊이 (mm)"),
    As: float = Field(description="인장 철근 단면적 (mm^2)"),
    fck: float = Field(description="콘크리트 압축강도 (MPa)"),
    fy: float = Field(description="철근 항복강도 (MPa)"),
    Mu: float = Field(description="소요 휨모멘트 (kN.m)"),
) -> str:
    a = As * fy / (0.85 * fck * b)
    Mn = As * fy * (d - a / 2) / 1e6
    phi_Mn = 0.85 * Mn
    rho = As / (b * d)
    rho_min = max(0.25 * math.sqrt(fck) / fy, 1.4 / fy)
    return json.dumps({
        "phi_Mn": round(phi_Mn, 2),
        "Mu": round(Mu, 2),
        "DCR": round(Mu / phi_Mn, 3),
        "flexure_check": "OK" if phi_Mn >= Mu else "NG",
        "rho": round(rho, 5),
        "rho_min": round(rho_min, 5),
    }, ensure_ascii=False, indent=2)

@mcp.tool(
    name="check_shear_strength",
    description="RC 부재 전단강도 검토 — KDS 41 17 00.",
)
def check_shear_strength(
    b: float = Field(description="보 폭 (mm)"),
    d: float = Field(description="유효 깊이 (mm)"),
    fck: float = Field(description="콘크리트 압축강도 (MPa)"),
    Av: float = Field(description="전단 보강근 면적 (mm^2)"),
    s: float = Field(description="전단 보강근 간격 (mm)"),
    fy: float = Field(description="전단 보강근 항복강도 (MPa)"),
    Vu: float = Field(description="소요 전단력 (kN)"),
) -> str:
    Vc = (1 / 6) * math.sqrt(fck) * b * d / 1000
    Vs = Av * fy * d / s / 1000
    phi_Vn = 0.75 * (Vc + Vs)
    return json.dumps({
        "phi_Vn": round(phi_Vn, 2),
        "Vu": round(Vu, 2),
        "DCR": round(Vu / phi_Vn, 3),
        "shear_check": "OK" if phi_Vn >= Vu else "NG",
    }, ensure_ascii=False, indent=2)

# ── (Stage 2 재현) Resource — KDS RAG ────────────────────────────
@mcp.resource("kds://41-17-00/summary", mime_type="application/json")
def kds_summary() -> str:
    return json.dumps({
        "title": "KDS 41 17 00 콘크리트구조 설계기준",
        "flexure": {"phi": 0.85, "rho_min": "max(0.25*sqrt(fck)/fy, 1.4/fy)"},
        "shear": {"phi": 0.75, "Vc": "(1/6)*sqrt(fck)*b*d", "Vs": "Av*fy*d/s"},
    }, ensure_ascii=False, indent=2)

print("Stage 1~3 reloaded: 2 tools + 1 resource.")


## §3. 미다스 모델 파일 형식 안내

미다스 시빌과 미다스 젠이 사용하는 모델 파일은 **텍스트 기반 명령 파일** 형식입니다. 파서가 알아야 할 핵심 규칙은 별표로 시작하는 줄이 새로운 섹션의 시작을 의미하고, 세미콜론으로 시작하는 줄은 주석이며, 그 외 줄은 콤마로 구분된 데이터 행이라는 점입니다. 이러한 텍스트 기반 형식은 사람이 직접 읽고 편집할 수 있어 협업과 형상 관리 측면에서 큰 장점을 가집니다.

```text
*UNIT       <- 섹션 헤더 (별표로 시작)
KN, M

*NODE       <- 절점 정의 섹션
; iNode, X, Y, Z   <- 주석 라인 (세미콜론)
1, 0.0, 0.0, 0.0   <- 콤마로 구분된 데이터 행
2, 6.0, 0.0, 0.0

*ELEMENT    <- 부재 정의 섹션
; iElem, TYPE, iMAT, iPRO, iN1, iN2
1, BEAM, 1, 1, 1, 2
```

각 섹션의 의미를 자세히 살펴보면 다음과 같습니다. 단위 섹션은 사용 단위계, 즉 힘의 단위와 길이의 단위를 지정하고, 절점 섹션은 절점 식별번호와 좌표를 차례로 적습니다. 부재 섹션은 부재의 연결 관계, 즉 어떤 절점과 어떤 절점을 잇는지와 어떤 재료와 어떤 단면을 사용하는지를 적습니다. 재료 섹션은 콘크리트나 강재 등 재료 종류와 강도 등급을 정의하고, 단면 섹션은 부재의 형상과 치수를 정의합니다. 하중 섹션은 다양한 하중 케이스를 담으며, 형식이 가장 다양한 섹션이기도 합니다.

> [!tip] 파서가 적용할 네 가지 분기 규칙
> 첫째, 별표로 시작하는 줄은 새로운 섹션의 시작이므로 현재 섹션 이름을 갱신합니다. 둘째, 세미콜론으로 시작하는 줄은 주석이므로 건너뜁니다. 셋째, 빈 줄도 건너뜁니다. 넷째, 그 외 모든 줄은 콤마로 분리한 뒤 현재 섹션의 데이터 행으로 누적합니다. 이 네 가지만 지키면 어떤 미다스 모델 파일이든 안전하게 파싱할 수 있습니다. 이러한 단순한 분기 규칙으로도 실무에서 사용하는 수천 줄 분량의 모델 파일을 무리 없이 처리할 수 있다는 점이 텍스트 기반 형식의 큰 장점입니다.


## §4. 검증용 샘플 모델 파일 만들기

파서 동작을 검증하기 위해 단순한 두 경간 보 모델, 즉 절점 세 개와 부재 두 개로 구성된 미니 모델을 텍스트로 직접 만들어 사용합니다. 실무에서 사용하는 모델은 수천 줄에 달하지만 형식 자체는 동일하므로 이 정도 크기로도 파서의 정확성을 충분히 검증할 수 있습니다.

In [ ]:
# Week_07.md §2.7 — 파서 검증용 미니 모델 (2-span 보, 단순 보강).
sample_mgt = """*UNIT
KN, M

*NODE
; iNode, X, Y, Z
1, 0.0, 0.0, 0.0
2, 6.0, 0.0, 0.0
3, 12.0, 0.0, 0.0

*ELEMENT
; iElem, TYPE, iMAT, iPRO, iN1, iN2
1, BEAM, 1, 1, 1, 2
2, BEAM, 1, 1, 2, 3

*MATERIAL
; iMAT, TYPE, NAME
1, CONC, C27

*SECTION
; iPRO, NAME, b, d
1, B300x600, 0.3, 0.6
"""

sample_path = Path("sample_model.mgt")
sample_path.write_text(sample_mgt, encoding="utf-8")
print(f"Sample saved: {sample_path.resolve()}  ({sample_path.stat().st_size} bytes)")


## §5. 파서 도구 구현하기

강의노트 §2.7 라인 1484에 명시된 표준 시그니처는 파일 경로 하나를 받아 요약 정보를 반환하는 형태입니다. 즉 사용자가 자신의 컴퓨터에 있는 어떤 미다스 모델 파일이든 절대경로로 지정하기만 하면, 파서가 해당 파일을 읽어들여 절점 개수와 부재 개수와 재료 정보 등을 한눈에 보여 줍니다.

구현 전략은 세 단계로 나누어 생각합니다. 먼저 입력받은 경로가 절대경로인지 검증합니다. 이는 운영 규칙에서 강조하는 'absolute paths only' 원칙에 부합합니다. 다음으로 파일을 줄 단위로 순회하면서 별표·세미콜론·빈 줄을 분기 처리하여 섹션별로 데이터를 누적합니다. 마지막으로 절점과 부재의 개수, 사용된 재료와 단면 정보, 그리고 발견된 모든 섹션 이름을 모아 한국어 식별자가 보존된 JSON 형태로 반환합니다.

이러한 구현 패턴은 클로드가 결과를 곧바로 한국어 자연어로 풀어낼 수 있게 해 주는 매우 중요한 디자인 결정입니다. 만약 영어 식별자만 반환했다면 클로드는 다시 한국어로 변환하는 작업을 수행해야 했을 것이고, 그 과정에서 도메인 용어의 일관성이 깨질 수 있었습니다. 한국 건축 실무에서 사용하는 절점, 부재, 재료, 단면이라는 용어를 그대로 보존하는 설계가 본 단계의 핵심 가치입니다.


In [ ]:
# Week_07.md §2.7 단계 ③ — parse_midas_mgt Tool
from typing import Dict, List

@mcp.tool(
    name="parse_midas_mgt",
    description=(
        "Parse a Midas .mgt model file and return a JSON summary of nodes, "
        "elements, materials, and sections. Input: absolute path to a .mgt file."
    ),
)
def parse_midas_mgt(
    file_path: str = Field(description="Absolute path to a Midas .mgt file"),
) -> str:
    p = Path(file_path)
    if not p.is_absolute():
        return json.dumps({"error": "file_path must be absolute", "given": file_path}, ensure_ascii=False)
    if not p.exists():
        return json.dumps({"error": "file not found", "given": str(p)}, ensure_ascii=False)

    sections: Dict[str, List[List[str]]] = {}
    current = None
    for raw in p.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith(";"):
            continue
        if line.startswith("*"):
            current = line[1:].split()[0].upper()
            sections.setdefault(current, [])
        elif current:
            cells = [c.strip() for c in line.split(",")]
            sections[current].append(cells)

    summary = {
        "file": str(p),
        "unit": sections.get("UNIT", []),
        "node_count": len(sections.get("NODE", [])),
        "element_count": len(sections.get("ELEMENT", [])),
        "materials": sections.get("MATERIAL", []),
        "sections": sections.get("SECTION", []),
        "raw_section_keys": sorted(sections.keys()),
    }
    return json.dumps(summary, ensure_ascii=False, indent=2)

print("Tool registered: parse_midas_mgt")


## §6. 도구 호출 검증하기

별도의 검사 도구(MCP Inspector) 클라이언트를 띄우지 않아도, FastMCP는 같은 파이썬 프로세스 내부에서 도구를 직접 호출할 수 있는 방법을 제공합니다. 본 셀에서는 `mcp.call_tool()` 호출을 비동기로 실행하여 결과를 살펴봅니다. **반드시 절대경로**로 호출해야 한다는 점에 다시 한번 주의합니다.

In [ ]:
# Week_07.md §2.7 — 도구 라우팅 확인. 절대경로로 호출하는 것이 핵심.
import asyncio

async def demo_parse():
    abs_path = str(Path("sample_model.mgt").resolve())
    print(f"Calling parse_midas_mgt with: {abs_path}\n")
    result = await mcp.call_tool("parse_midas_mgt", {"file_path": abs_path})
    # FastMCP returns a list of TextContent
    text = result[0].text if isinstance(result, list) else result
    print(text)

await demo_parse()


## §7. 클로드와 결합한 자연어 시나리오

이제 클로드는 다음과 같은 자연어 요청을 받으면 자동으로 미다스 파서 도구를 호출할 수 있습니다.

> 사용자가 묻기를 *"방금 받은 샘플 모델 파일의 부재 개수와 사용된 콘크리트 강도를 알려줘."*

이 요청에 대한 클로드의 동작은 다음과 같이 자동으로 진행됩니다. 먼저 클로드는 자신이 사용 가능한 도구 목록을 조회하여 미다스 파서 도구를 발견합니다. 다음으로 사용자가 언급한 파일의 절대경로를 인자로 채워 해당 도구를 호출합니다. 그러면 서버는 파일을 읽어 부재 개수와 사용된 재료 정보를 한국어 식별자가 보존된 JSON 형태로 반환합니다. 마지막으로 클로드는 그 결과를 자연스러운 한국어 문장으로 풀어내어 사용자에게 답변합니다.

```mermaid
sequenceDiagram
    participant 사용자
    participant 클로드 as 클로드(LLM)
    participant 서버 as structural-mcp 서버
    사용자->>클로드: "부재 개수와 콘크리트 강도?"
    클로드->>서버: 도구 목록 조회
    서버-->>클로드: 미다스 파서 도구 발견
    클로드->>서버: 미다스 파서 도구 호출 (절대경로 전달)
    서버-->>클로드: 부재 두 개, C27 콘크리트 등 요약
    클로드-->>사용자: "부재는 두 개이고, C27 콘크리트가 사용되었습니다."
```

> [!ref] 강의노트와 스킬자 7번 레슨의 메시지
> 외부 자원을 *동적으로 읽어들이는 동작*은 도구로 노출하면, 클로드가 자율적으로 호출하여 분석을 이어갈 수 있습니다. 이것이 **모델 컨텍스트 프로토콜이 단순한 검색 증강 생성보다 강력한 이유** 중 하나입니다. 검색 증강 생성은 검색만 가능하지만, 모델 컨텍스트 프로토콜의 도구는 *임의의 외부 시스템과 상호작용* 할 수 있어 한국 구조설계 사무소의 실제 업무 흐름에 매우 자연스럽게 녹아들 수 있습니다.


## §8. 통합본 저장하기 — 도구와 리소스와 파서 한꺼번에

누적 빌드업 원칙에 따라, 단계 4의 산출물을 파이썬 파일로 다시 내보내 다음 단계에서 그대로 가져다 쓸 수 있게 합니다. 이렇게 해두면 노트북 사이에 코드를 중복으로 정의할 필요 없이 단계별 학습이 가능해집니다.

In [ ]:
# 누적 빌드업: Stage 4의 산출물을 .py로 내보내 다음 단계(S6_st05)에서 import.
structural_mcp_py = '''"""structural-mcp — Stage 4 (Tools + Resources + Midas .mgt parser).

Cumulative build-up of Week_07.md §2.7:
  Stage 1: FastMCP init
  Stage 2: KDS RAG resource
  Stage 3: Strength check tools (flexural, shear)
  Stage 4: Midas .mgt parser tool   <-- this file
"""
import json
import math
from pathlib import Path
from typing import Dict, List

from pydantic import Field
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("StructuralMCP", log_level="ERROR")


@mcp.tool(
    name="check_flexural_strength",
    description="RC 보 휨강도 검토 (KDS 41 17 00; SI: mm/MPa/kN.m).",
)
def check_flexural_strength(
    b: float = Field(description="보 폭 (mm)"),
    d: float = Field(description="유효 깊이 (mm)"),
    As: float = Field(description="인장 철근 단면적 (mm^2)"),
    fck: float = Field(description="콘크리트 압축강도 (MPa)"),
    fy: float = Field(description="철근 항복강도 (MPa)"),
    Mu: float = Field(description="소요 휨모멘트 (kN.m)"),
) -> str:
    a = As * fy / (0.85 * fck * b)
    Mn = As * fy * (d - a / 2) / 1e6
    phi_Mn = 0.85 * Mn
    rho = As / (b * d)
    rho_min = max(0.25 * math.sqrt(fck) / fy, 1.4 / fy)
    return json.dumps({
        "phi_Mn": round(phi_Mn, 2), "Mu": round(Mu, 2),
        "DCR": round(Mu / phi_Mn, 3),
        "flexure_check": "OK" if phi_Mn >= Mu else "NG",
        "rho": round(rho, 5), "rho_min": round(rho_min, 5),
    }, ensure_ascii=False, indent=2)


@mcp.tool(
    name="check_shear_strength",
    description="RC 부재 전단강도 검토 (KDS 41 17 00).",
)
def check_shear_strength(
    b: float = Field(description="보 폭 (mm)"),
    d: float = Field(description="유효 깊이 (mm)"),
    fck: float = Field(description="콘크리트 압축강도 (MPa)"),
    Av: float = Field(description="전단 보강근 면적 (mm^2)"),
    s: float = Field(description="전단 보강근 간격 (mm)"),
    fy: float = Field(description="전단 보강근 항복강도 (MPa)"),
    Vu: float = Field(description="소요 전단력 (kN)"),
) -> str:
    Vc = (1 / 6) * math.sqrt(fck) * b * d / 1000
    Vs = Av * fy * d / s / 1000
    phi_Vn = 0.75 * (Vc + Vs)
    return json.dumps({
        "phi_Vn": round(phi_Vn, 2), "Vu": round(Vu, 2),
        "DCR": round(Vu / phi_Vn, 3),
        "shear_check": "OK" if phi_Vn >= Vu else "NG",
    }, ensure_ascii=False, indent=2)


@mcp.tool(
    name="parse_midas_mgt",
    description=(
        "Parse a Midas .mgt model file and return a JSON summary of nodes, "
        "elements, materials, and sections."
    ),
)
def parse_midas_mgt(
    file_path: str = Field(description="Absolute path to a Midas .mgt file"),
) -> str:
    p = Path(file_path)
    if not p.is_absolute():
        return json.dumps({"error": "file_path must be absolute"}, ensure_ascii=False)
    if not p.exists():
        return json.dumps({"error": "file not found", "given": str(p)}, ensure_ascii=False)

    sections: Dict[str, List[List[str]]] = {}
    current = None
    for raw in p.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith(";"):
            continue
        if line.startswith("*"):
            current = line[1:].split()[0].upper()
            sections.setdefault(current, [])
        elif current:
            sections[current].append([c.strip() for c in line.split(",")])

    return json.dumps({
        "file": str(p),
        "unit": sections.get("UNIT", []),
        "node_count": len(sections.get("NODE", [])),
        "element_count": len(sections.get("ELEMENT", [])),
        "materials": sections.get("MATERIAL", []),
        "sections": sections.get("SECTION", []),
        "raw_section_keys": sorted(sections.keys()),
    }, ensure_ascii=False, indent=2)


@mcp.resource("kds://41-17-00/summary", mime_type="application/json")
def kds_summary() -> str:
    return json.dumps({
        "title": "KDS 41 17 00 콘크리트구조 설계기준",
        "flexure": {"phi": 0.85, "rho_min": "max(0.25*sqrt(fck)/fy, 1.4/fy)"},
        "shear": {"phi": 0.75, "Vc": "(1/6)*sqrt(fck)*b*d", "Vs": "Av*fy*d/s"},
    }, ensure_ascii=False, indent=2)


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

Path("structural_mcp.py").write_text(structural_mcp_py, encoding="utf-8")
print(f"Saved: {Path('structural_mcp.py').resolve()}")
print(f"Size:  {Path('structural_mcp.py').stat().st_size} bytes")


## §9. 다음 단계 안내 — 단계 5 (프롬프트로 도메인 워크플로 외부화)

이제 도구 세 개(휨 검토·전단 검토·미다스 파서)와 리소스 한 개(KDS 요약)를 갖췄습니다. **그러나 사용자는 매번 다음과 같은 긴 지시를 입력해야 하는 불편함이 남아 있습니다**.

> *"먼저 KDS 요약 리소스를 읽고, KDS 4.3.1 인용 형식으로 답하고, 단위는 SI로 통일하고..."*

다음 노트북인 단계 5에서는 이 반복되는 도메인 규약을 **`structural_review` 프롬프트** 한 줄로 외부화합니다. 이는 강의노트 §2.7 단계 ④에서 다루는 **도구·리소스·프롬프트 세 가지 컴포넌트의 협업** 패턴의 핵심입니다.

이번 단계의 의의를 다시 정리하면 다음과 같습니다. 첫째, 외부 소프트웨어의 결과 파일을 클로드가 직접 읽을 수 있도록 도구로 노출하는 패턴을 익혔습니다. 둘째, 절대경로 검증과 라인 단위 분기 처리와 한국어 식별자 보존이라는 세 가지 원칙을 한 자리에서 적용해 보았습니다. 셋째, 누적 빌드업 방식을 통해 노트북 간에 코드 중복 없이 단계별로 학습할 수 있는 구조를 확인했습니다.

> [!ref] 강의노트 §2.7 단계 ④ (라인 1531-1554)
> 구조 검토용 프롬프트는 도구·리소스·파서 세 가지를 한 번에 호출하는 **도메인 워크플로 캡슐**로 동작합니다. 사용자는 인자만 채우면 되고, KDS 조항 인용·SI 단위 강제·소요/공칭 강도비(DCR) 산정 등 도메인 규약은 프롬프트 내부에서 자동으로 강제됩니다. 이 패턴은 한국 건축 실무의 도메인 규약이 많고 정형화되어 있는 특성과 매우 잘 맞으며, 학생과 실무자가 같은 형식의 검토 결과를 자연어로 공유할 수 있게 해줍니다.

> [!action] 다음 노트북으로 이동하기 전에 점검할 사항
> 본 노트북에서 저장한 `structural_mcp.py` 파일이 같은 디렉터리에 정상적으로 만들어졌는지 확인해 주세요. 파일 크기가 비정상적으로 작으면 단계 5에서 import 오류가 발생할 수 있습니다.
